# Threading Diagnostics and Sensitivity Analysis

Purpose: evaluate the empirical timing patterns that support conversation-thread construction and create threaded message datasets using alternative inactivity cutoffs.

This notebook starts from the pre-thread message-level dataset created in `01_build_core_message_preprocessing_clean.ipynb`. It computes same-patient, same-key intermessage gaps, builds threads using 15-day, 30-day, and 60-day cutoffs, and saves the 30-day threaded dataset as the primary downstream handoff.

## Scope

This notebook does:

- load `messages_without_threads.parquet`
- calculate intermessage gap summaries using the threading composite key
- construct threads using 15-day, 30-day, and 60-day inactivity thresholds
- compare thread characteristics across thresholds
- save the primary 30-day threaded handoff as `messages_with_threads.parquet`
- write aggregate QA and reproducibility summaries

This notebook does not:

- merge demographics, CCI, or encounter covariates
- create the final analysis dataset
- run regression models
- display row-level patient or message examples

## Downstream Handoff

The canonical output `messages_with_threads.parquet` is used by `05_build_exposure_analysis_dataset_clean.ipynb` to construct thread-level exposures and outcomes.


## 1. Setup

Configure input/output paths, required columns, and sensitivity cutoffs. The input is the finalized cleaned message-level dataset before thread assignment.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import platform
import sys
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.max_columns = 120


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
RUNS_DIR = OUTPUT_ROOT / "runs"
WORKING_OUTPUT_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()

RUNS_DIR.mkdir(parents=True, exist_ok=True)
WORKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CUTOFF_DAYS = [15, 30, 60]
GAP_PLOT_CAP_DAYS = 120

REQUIRED_COLUMNS = [
    "DMRN",
    "MESSAGE_ID",
    "CREATED_TIME",
    "SUBJECT_CLEAN",
    "Prov_DEPT_For_Appt",
    "MESG_TYPE",
    "Sender_all",
    "Receiver_all",
    "origin",
]

OPTIONAL_COLUMNS = [
    "SENDER_TYPE",
    "RECEIVER_TYPE",
    "response_time_hours",
    "update_delay_hours",
]

INPUT_PATH = WORKING_OUTPUT_DIR / "messages_without_threads.parquet"
CANONICAL_THREADED_MESSAGES_PATH = WORKING_OUTPUT_DIR / "messages_with_threads.parquet"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUNS_DIR / f"threading_diagnostics_sensitivity_{RUN_ID}"
if RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists: {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("REPO_ROOT:", REPO_ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("RUN_DIR:", RUN_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("WORKING_OUTPUT_DIR:", WORKING_OUTPUT_DIR)
print("CANONICAL_THREADED_MESSAGES_PATH:", CANONICAL_THREADED_MESSAGES_PATH)
print("CUTOFF_DAYS:", CUTOFF_DAYS)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file does not exist: {INPUT_PATH}\n"
        "Run 01_build_core_message_preprocessing_clean.ipynb first."
    )

## 2. Load Data

Load the finalized cleaned message-level dataset and verify required columns before diagnostics or threading.

In [ ]:
messages = pd.read_parquet(INPUT_PATH)
messages["CREATED_TIME"] = pd.to_datetime(messages["CREATED_TIME"], errors="coerce")

column_availability = pd.DataFrame(
    {
        "column": REQUIRED_COLUMNS + OPTIONAL_COLUMNS,
        "required": [True] * len(REQUIRED_COLUMNS) + [False] * len(OPTIONAL_COLUMNS),
        "available": [(col in messages.columns) for col in REQUIRED_COLUMNS + OPTIONAL_COLUMNS],
    }
)
missing_required = column_availability.loc[column_availability["required"] & ~column_availability["available"], "column"].tolist()

print("Message-level dataset shape:", messages.shape)
display(column_availability)

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

origin_counts = (
    messages["origin"]
    .value_counts(dropna=False)
    .rename_axis("origin")
    .reset_index(name="messages")
)
origin_counts["percent"] = (origin_counts["messages"] / len(messages) * 100).round(3)
display(origin_counts)

## 3. Gap Diagnostics

Gaps are calculated between consecutive messages with the same composite key: `DMRN`, `SUBJECT_CLEAN`, `Prov_DEPT_For_Appt`, and `MESG_TYPE`. The diagnostic summaries below support the 30-day primary rule and the 15-day/60-day sensitivity analyses.

In [ ]:
def normalize_thread_key(series: pd.Series, missing_label: str) -> pd.Series:
    return series.astype("string").str.strip().str.lower().replace("", pd.NA).fillna(missing_label)


def summarize_gap_days(data: pd.DataFrame, group_cols=None) -> pd.DataFrame:
    group_cols = group_cols or []
    pct = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

    def summarize_one(x: pd.Series) -> pd.Series:
        x = x.dropna()
        q = x.quantile(pct) if len(x) else pd.Series(index=pct, dtype=float)
        return pd.Series(
            {
                "n_gaps": int(x.size),
                "median_days": q.loc[0.50] if len(x) else np.nan,
                "q25_days": q.loc[0.25] if len(x) else np.nan,
                "q75_days": q.loc[0.75] if len(x) else np.nan,
                "iqr_days": (q.loc[0.75] - q.loc[0.25]) if len(x) else np.nan,
                "p90_days": q.loc[0.90] if len(x) else np.nan,
                "p95_days": q.loc[0.95] if len(x) else np.nan,
                "p99_days": q.loc[0.99] if len(x) else np.nan,
                "pct_gaps_le_15d": (x.le(15).mean() * 100) if len(x) else np.nan,
                "pct_gaps_le_30d": (x.le(30).mean() * 100) if len(x) else np.nan,
                "pct_gaps_le_60d": (x.le(60).mean() * 100) if len(x) else np.nan,
            }
        )

    if group_cols:
        return data.groupby(group_cols, dropna=False)["gap_days"].apply(summarize_one).unstack().reset_index()
    return summarize_one(data["gap_days"]).to_frame().T


gap_work = messages.copy()
gap_work["THREAD_SUBJECT_KEY"] = normalize_thread_key(gap_work["SUBJECT_CLEAN"], "unknown_subject")
gap_work["THREAD_DEPARTMENT_KEY"] = normalize_thread_key(gap_work["Prov_DEPT_For_Appt"], "unknown_department")
gap_work["THREAD_MESSAGE_TYPE_KEY"] = normalize_thread_key(gap_work["MESG_TYPE"], "unknown_message_type")

thread_key_cols = ["DMRN", "THREAD_SUBJECT_KEY", "THREAD_DEPARTMENT_KEY", "THREAD_MESSAGE_TYPE_KEY"]

# Match the original threading logic: process messages chronologically within patient.
# The composite key is used for grouping, not for chronological sorting.
gap_work = gap_work.sort_values(["DMRN", "CREATED_TIME"], kind="mergesort")
gap_work["previous_same_key_time"] = gap_work.groupby(thread_key_cols, sort=False, dropna=False)["CREATED_TIME"].shift(1)
gap_work["previous_origin"] = gap_work.groupby(thread_key_cols, sort=False, dropna=False)["origin"].shift(1)
gap_work["gap_days"] = (gap_work["CREATED_TIME"] - gap_work["previous_same_key_time"]).dt.total_seconds().div(86400)
gap_work["origin_transition_pair"] = gap_work["previous_origin"].astype("string") + " -> " + gap_work["origin"].astype("string")

gaps = gap_work.dropna(subset=["gap_days"]).copy()

print("Number of same-key gaps:", len(gaps))

In [ ]:
overall_gap_summary = summarize_gap_days(gaps)
gap_summary_by_message_type = summarize_gap_days(gaps, ["MESG_TYPE"]).sort_values("n_gaps", ascending=False)
gap_summary_by_origin_transition = summarize_gap_days(gaps, ["origin_transition_pair"]).sort_values("n_gaps", ascending=False)

overall_gap_summary_path = RUN_DIR / "gap_summary_overall.csv"
gap_summary_by_message_type_path = RUN_DIR / "gap_summary_by_message_type.csv"
gap_summary_by_origin_transition_path = RUN_DIR / "gap_summary_by_origin_transition.csv"

overall_gap_summary.to_csv(overall_gap_summary_path, index=False)
gap_summary_by_message_type.to_csv(gap_summary_by_message_type_path, index=False)
gap_summary_by_origin_transition.to_csv(gap_summary_by_origin_transition_path, index=False)

print("Overall gap summary:")
display(overall_gap_summary)
print("Gap summary by message type:")
display(gap_summary_by_message_type)
print("Gap summary by origin transition pair:")
display(gap_summary_by_origin_transition)

In [ ]:
gap_values_capped = gaps.loc[gaps["gap_days"].between(0, GAP_PLOT_CAP_DAYS), "gap_days"].dropna()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(gap_values_capped, bins=60, color="#4C78A8", edgecolor="white")
for cutoff in CUTOFF_DAYS:
    ax.axvline(cutoff, linestyle="--", linewidth=1.5, label=f"{cutoff} days")
ax.set_title(f"Same-key inter-message gaps, capped at {GAP_PLOT_CAP_DAYS} days")
ax.set_xlabel("Gap days")
ax.set_ylabel("Number of gaps")
ax.legend()
fig.tight_layout()
histogram_path = RUN_DIR / "gap_days_histogram_capped.png"
fig.savefig(histogram_path, dpi=300, bbox_inches="tight")
plt.show()

sorted_gap_values = np.sort(gap_values_capped.to_numpy())
ecdf = np.arange(1, len(sorted_gap_values) + 1) / len(sorted_gap_values) if len(sorted_gap_values) else np.array([])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sorted_gap_values, ecdf, color="#2F4B7C", linewidth=2)
for cutoff in CUTOFF_DAYS:
    ax.axvline(cutoff, linestyle="--", linewidth=1.5, label=f"{cutoff} days")
ax.set_title(f"ECDF of same-key inter-message gaps, capped at {GAP_PLOT_CAP_DAYS} days")
ax.set_xlabel("Gap days")
ax.set_ylabel("Cumulative proportion")
ax.set_ylim(0, 1.01)
ax.legend()
fig.tight_layout()
ecdf_path = RUN_DIR / "gap_days_ecdf_capped.png"
fig.savefig(ecdf_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved gap figures:")
print(" -", histogram_path)
print(" -", ecdf_path)

## 4. Thread Construction Function

`build_threads(messages, cutoff_days)` assigns message-level thread identifiers and creates a thread-level summary. The primary rule is 30 days; 15-day and 60-day cutoffs are sensitivity analyses.

### Conversation-Segmentation Logic

The thread-building function treats a conversation as a sequence of messages from the same patient with the same normalized subject, department, and message type, separated by no more than the selected inactivity cutoff. The primary analysis uses a 30-day cutoff; 15-day and 60-day cutoffs are constructed as sensitivity analyses.

#### Textual Flow Map

1. **Input message-level data**

   Required fields:

   - `DMRN`
   - `MESSAGE_ID`
   - `CREATED_TIME`
   - `SUBJECT_CLEAN`
   - `Prov_DEPT_For_Appt`
   - `MESG_TYPE`

2. **Normalize the threading key**

   A composite key is created for each message:

   ```text
   patient + normalized subject + normalized department + normalized message type
   ```

   In code, this corresponds to:

   - `DMRN`
   - `THREAD_SUBJECT_KEY`
   - `THREAD_DEPARTMENT_KEY`
   - `THREAD_MESSAGE_TYPE_KEY`

3. **Sort messages in temporal order**

   Messages are ordered by:

   ```text
   DMRN -> CREATED_TIME
   ```

   This preserves the original patient-time ordering used for conversation reconstruction.

4. **Calculate the previous same-key message time**

   Within each composite key, the previous message timestamp is carried forward:

   ```text
   previous_same_key_time = prior CREATED_TIME within the same composite key
   gap_days_same_key = CREATED_TIME - previous_same_key_time
   ```

5. **Start or continue a thread**

   For each message:

   ```text
   if this is the first message for the composite key:
       start a new thread
   elif gap_days_same_key > cutoff_days:
       start a new thread
   else:
       continue the existing thread for that composite key
   ```

6. **Assign thread identifiers**

   Consecutive same-key segments are numbered and mapped back to the full message-level dataset as `thread_id`.

7. **Create message- and thread-level outputs**

   The function returns:

   - a message-level dataset with `thread_id`, `starts_new_thread`, and `gap_days_same_key`
   - a thread-level summary dataset
   - a QA dictionary for row counts, thread counts, and thread characteristics

#### Key Interpretation Points

- A change in subject, department, or message type creates a different composite key and therefore cannot continue the same thread.
- A same-key message continues the prior thread only if the inactivity gap is within the selected cutoff.
- A same-key message after a long inactive period starts a new thread, preventing very old conversations from remaining open indefinitely.
- Interleaved workflows are handled because the gap is calculated within each patient-specific composite key, not merely between adjacent messages overall.
- The 30-day threshold is the primary definition; 15-day and 60-day versions are used to evaluate sensitivity to the inactivity cutoff.


In [ ]:
def unique_join(series: pd.Series) -> str:
    values = pd.Series(series.dropna().astype(str).unique())
    values = values[values.str.strip().ne("")]
    return "; ".join(sorted(values.tolist()))


def build_threads(messages: pd.DataFrame, cutoff_days: int):
    required = [
        "DMRN", "MESSAGE_ID", "CREATED_TIME", "SUBJECT_CLEAN",
        "Prov_DEPT_For_Appt", "MESG_TYPE", "Sender_all", "Receiver_all", "origin",
    ]
    missing = [col for col in required if col not in messages.columns]
    if missing:
        raise ValueError(f"Missing required columns for threading: {missing}")

    thread_cutoff = pd.Timedelta(days=cutoff_days)
    out = messages.copy()
    out["CREATED_TIME"] = pd.to_datetime(out["CREATED_TIME"], errors="coerce")

    out["THREAD_SUBJECT_KEY"] = normalize_thread_key(out["SUBJECT_CLEAN"], "unknown_subject")
    out["THREAD_DEPARTMENT_KEY"] = normalize_thread_key(out["Prov_DEPT_For_Appt"], "unknown_department")
    out["THREAD_MESSAGE_TYPE_KEY"] = normalize_thread_key(out["MESG_TYPE"], "unknown_message_type")

    thread_key_cols = ["DMRN", "THREAD_SUBJECT_KEY", "THREAD_DEPARTMENT_KEY", "THREAD_MESSAGE_TYPE_KEY"]

    # Match the original threading logic: walk forward by patient and CREATED_TIME.
    # MESSAGE_ID is retained as an identifier, but it is not used to order messages or define threads.
    thread_work = (
        out.reset_index(names="source_row_order")
        [["source_row_order", "MESSAGE_ID", "CREATED_TIME"] + thread_key_cols]
        .sort_values(["DMRN", "CREATED_TIME"], kind="mergesort")
        .copy()
    )

    thread_work["previous_same_key_time"] = thread_work.groupby(thread_key_cols, sort=False, dropna=False)["CREATED_TIME"].shift(1)
    thread_work["gap_days_same_key"] = (
        thread_work["CREATED_TIME"].sub(thread_work["previous_same_key_time"]).dt.total_seconds().div(86400)
    )
    thread_work["starts_new_thread"] = (
        thread_work["previous_same_key_time"].isna()
        | (thread_work["CREATED_TIME"].sub(thread_work["previous_same_key_time"]) > thread_cutoff)
    )
    thread_work["thread_segment_within_key"] = (
        thread_work.groupby(thread_key_cols, sort=False, dropna=False)["starts_new_thread"].cumsum().astype("int64")
    )

    segment_cols = thread_key_cols + ["thread_segment_within_key"]
    thread_lookup = (
        thread_work.groupby(segment_cols, sort=False, dropna=False)
        .agg(thread_start_time=("CREATED_TIME", "min"), first_source_row_order=("source_row_order", "min"))
        .reset_index()
        .sort_values(["thread_start_time", "first_source_row_order"], kind="mergesort")
        .reset_index(drop=True)
    )
    thread_lookup["thread_id"] = np.arange(1, len(thread_lookup) + 1, dtype="int64")

    assignments = thread_work.merge(
        thread_lookup[segment_cols + ["thread_id"]],
        on=segment_cols,
        how="left",
        validate="many_to_one",
    )[["source_row_order", "thread_id", "previous_same_key_time", "gap_days_same_key", "starts_new_thread"]]

    out = (
        out.reset_index(names="source_row_order")
        .merge(assignments, on="source_row_order", how="left", validate="one_to_one")
        .drop(columns=["source_row_order"])
    )
    out["thread_id"] = out["thread_id"].astype("Int64")

    ordered_index = out.sort_values(["thread_id", "CREATED_TIME"], kind="mergesort").index
    ordered_for_sequence = out.loc[ordered_index].copy()
    ordered_for_sequence["thread_sequence_in_thread"] = (
        ordered_for_sequence.groupby("thread_id", sort=False, dropna=False)
        .cumcount()
        .add(1)
        .astype("Int64")
    )

    out["thread_sequence_in_thread"] = pd.Series(pd.NA, index=out.index, dtype="Int64")
    out.loc[ordered_for_sequence.index, "thread_sequence_in_thread"] = ordered_for_sequence["thread_sequence_in_thread"]

    messages_for_threads = ordered_for_sequence

    optional_aggs = {}
    if "SENDER_TYPE" in messages_for_threads.columns:
        optional_aggs["INITIATOR_SENDER_TYPE"] = ("SENDER_TYPE", "first")
    if "RECEIVER_TYPE" in messages_for_threads.columns:
        optional_aggs["FIRST_RECEIVER_TYPE"] = ("RECEIVER_TYPE", "first")
    if "response_time_hours" in messages_for_threads.columns:
        optional_aggs["MED_RESPONSE_TIME_HOURS"] = ("response_time_hours", "median")
    if "update_delay_hours" in messages_for_threads.columns:
        optional_aggs["MED_UPDATE_DELAY_HOURS"] = ("update_delay_hours", "median")

    threads = (
        messages_for_threads.groupby("thread_id", sort=False, dropna=False)
        .agg(
            DMRN=("DMRN", "first"),
            SUBJECT_CLEAN=("SUBJECT_CLEAN", "first"),
            Prov_DEPT_For_Appt=("Prov_DEPT_For_Appt", "first"),
            MESG_TYPE=("MESG_TYPE", "first"),
            THREAD_SUBJECT_KEY=("THREAD_SUBJECT_KEY", "first"),
            THREAD_DEPARTMENT_KEY=("THREAD_DEPARTMENT_KEY", "first"),
            THREAD_MESSAGE_TYPE_KEY=("THREAD_MESSAGE_TYPE_KEY", "first"),
            START_TIME=("CREATED_TIME", "min"),
            END_TIME=("CREATED_TIME", "max"),
            MESSAGE_COUNT=("MESSAGE_ID", "size"),
            FIRST_MESSAGE_ID=("MESSAGE_ID", "first"),
            LAST_MESSAGE_ID=("MESSAGE_ID", "last"),
            INITIATOR_CATEGORY=("Sender_all", "first"),
            INITIATOR_ORIGIN=("origin", "first"),
            FIRST_RECEIVER_CATEGORY=("Receiver_all", "first"),
            SENDERS_LIST=("Sender_all", unique_join),
            RECEIVERS_LIST=("Receiver_all", unique_join),
            **optional_aggs,
        )
        .reset_index()
    )

    threads["DURATION_HOURS"] = threads["END_TIME"].sub(threads["START_TIME"]).dt.total_seconds().div(3600)
    threads["DURATION_DAYS"] = threads["DURATION_HOURS"].div(24)
    threads["IS_SINGLE_MESSAGE_THREAD"] = threads["MESSAGE_COUNT"].eq(1).astype(int)
    threads["IS_PROVIDER_INITIATED_THREAD"] = threads["INITIATOR_ORIGIN"].eq("Provider").astype(int)

    sender_counts = pd.crosstab(out["thread_id"], out["Sender_all"]).add_prefix("SEND_").add_suffix("_COUNT")
    origin_counts = pd.crosstab(out["thread_id"], out["origin"]).add_prefix("ORIGIN_").add_suffix("_COUNT")
    threads = threads.merge(sender_counts, left_on="thread_id", right_index=True, how="left")
    threads = threads.merge(origin_counts, left_on="thread_id", right_index=True, how="left")

    qa = {
        "cutoff_days": int(cutoff_days),
        "message_rows": int(len(out)),
        "thread_rows": int(len(threads)),
        "missing_thread_id_rows": int(out["thread_id"].isna().sum()),
        "single_message_threads": int(threads["IS_SINGLE_MESSAGE_THREAD"].sum()),
        "multi_message_threads": int((~threads["IS_SINGLE_MESSAGE_THREAD"].astype(bool)).sum()),
        "median_messages_per_thread": float(threads["MESSAGE_COUNT"].median()),
        "median_thread_duration_days": float(threads["DURATION_DAYS"].median()),
    }
    return out, threads, qa

## 5. Run Thread Construction

Run the same function for 15-day, 30-day, and 60-day inactivity cutoffs. Each message-level and thread-level dataset is saved separately.

In [ ]:
thread_results = {}
thread_qa_rows = []

for cutoff in CUTOFF_DAYS:
    print(f"Building threads with {cutoff}-day cutoff")
    messages_threaded, threads, qa = build_threads(messages, cutoff)

    messages_path = RUN_DIR / f"messages_threaded_{cutoff}d.parquet"
    threads_path = RUN_DIR / f"threads_{cutoff}d.parquet"
    qa_path = RUN_DIR / f"threading_qa_{cutoff}d.json"

    messages_threaded.to_parquet(messages_path, index=False, compression="zstd")
    threads.to_parquet(threads_path, index=False, compression="zstd")
    with open(qa_path, "w", encoding="utf-8") as f:
        json.dump(qa, f, indent=2, default=str)

    qa.update(
        {
            "messages_threaded_file": str(messages_path),
            "threads_file": str(threads_path),
            "qa_file": str(qa_path),
        }
    )
    thread_results[cutoff] = {"messages": messages_threaded, "threads": threads, "qa": qa}
    thread_qa_rows.append(qa)
    display(pd.DataFrame([qa]))

thread_qa_summary = pd.DataFrame(thread_qa_rows)
thread_qa_summary_path = RUN_DIR / "threading_qa_summary.csv"
thread_qa_summary.to_csv(thread_qa_summary_path, index=False)
display(thread_qa_summary)

## 6. Thread Comparison Diagnostics

Compare thread counts, message counts, duration, and initiating origin across the three inactivity cutoffs.

In [ ]:
def pct(numerator, denominator):
    return float(numerator / denominator * 100) if denominator else np.nan


def summarize_threads_for_cutoff(cutoff: int, threads: pd.DataFrame) -> dict:
    total_threads = len(threads)
    single_n = int(threads["IS_SINGLE_MESSAGE_THREAD"].sum())
    multi_n = int(total_threads - single_n)
    initiator_counts = threads["INITIATOR_ORIGIN"].value_counts(dropna=False)
    q25 = threads["MESSAGE_COUNT"].quantile(0.25)
    q75 = threads["MESSAGE_COUNT"].quantile(0.75)

    return {
        "cutoff_days": cutoff,
        "total_messages": int(threads["MESSAGE_COUNT"].sum()),
        "total_threads": int(total_threads),
        "median_messages_per_thread": float(threads["MESSAGE_COUNT"].median()),
        "q25_messages_per_thread": float(q25),
        "q75_messages_per_thread": float(q75),
        "iqr_messages_per_thread": float(q75 - q25),
        "single_message_threads_n": single_n,
        "single_message_threads_pct": pct(single_n, total_threads),
        "multi_message_threads_n": multi_n,
        "multi_message_threads_pct": pct(multi_n, total_threads),
        "median_thread_duration_days": float(threads["DURATION_DAYS"].median()),
        "p95_thread_duration_days": float(threads["DURATION_DAYS"].quantile(0.95)),
        "patient_initiated_threads_n": int(initiator_counts.get("Patient", 0)),
        "patient_initiated_threads_pct": pct(int(initiator_counts.get("Patient", 0)), total_threads),
        "provider_hcw_initiated_threads_n": int(initiator_counts.get("Provider", 0)),
        "provider_hcw_initiated_threads_pct": pct(int(initiator_counts.get("Provider", 0)), total_threads),
        "system_initiated_threads_n": int(initiator_counts.get("System", 0)),
        "system_initiated_threads_pct": pct(int(initiator_counts.get("System", 0)), total_threads),
    }

threading_sensitivity_summary = pd.DataFrame(
    [summarize_threads_for_cutoff(cutoff, result["threads"]) for cutoff, result in thread_results.items()]
).sort_values("cutoff_days")

threading_sensitivity_summary_path = RUN_DIR / "threading_sensitivity_summary.csv"
threading_sensitivity_summary.to_csv(threading_sensitivity_summary_path, index=False)
display(threading_sensitivity_summary)

## 7. Comparison Versus 30-Day Primary Rule

Compare the 15-day and 60-day sensitivity cutoffs with the 30-day primary threading rule.

In [ ]:
primary_cutoff = 30
primary_row = threading_sensitivity_summary.loc[threading_sensitivity_summary["cutoff_days"].eq(primary_cutoff)]
if primary_row.empty:
    raise ValueError("The 30-day primary cutoff is missing from threading_sensitivity_summary.")
primary_row = primary_row.iloc[0]

comparison_rows = []
numeric_metrics = [
    col for col in threading_sensitivity_summary.columns
    if col != "cutoff_days" and pd.api.types.is_numeric_dtype(threading_sensitivity_summary[col])
]

for _, row in threading_sensitivity_summary.loc[~threading_sensitivity_summary["cutoff_days"].eq(primary_cutoff)].iterrows():
    for metric in numeric_metrics:
        value = row[metric]
        primary_value = primary_row[metric]
        absolute_difference = value - primary_value
        percent_difference = (absolute_difference / primary_value * 100) if pd.notna(primary_value) and primary_value != 0 else np.nan
        comparison_rows.append(
            {
                "comparison_cutoff_days": int(row["cutoff_days"]),
                "primary_cutoff_days": primary_cutoff,
                "metric": metric,
                "comparison_value": value,
                "primary_30d_value": primary_value,
                "absolute_difference": absolute_difference,
                "percent_difference": percent_difference,
            }
        )

threading_sensitivity_comparison_vs_30d = pd.DataFrame(comparison_rows)
comparison_vs_30d_path = RUN_DIR / "threading_sensitivity_comparison_vs_30d.csv"
threading_sensitivity_comparison_vs_30d.to_csv(comparison_vs_30d_path, index=False)
display(threading_sensitivity_comparison_vs_30d)

## 8. Analytic-Relevant Diagnostics

These diagnostics summarize provider/HCW initiation and patient reply patterns when required columns are available. If a role or reply column is unavailable, the notebook records placeholders and prints the missing columns.

In [ ]:
def analytic_relevant_diagnostics_for_cutoff(cutoff: int, messages_threaded: pd.DataFrame, threads: pd.DataFrame) -> dict:
    diagnostics = {"cutoff_days": cutoff}
    missing_notes = []

    total_threads = len(threads)
    provider_threads = threads["INITIATOR_ORIGIN"].eq("Provider") if "INITIATOR_ORIGIN" in threads.columns else pd.Series(False, index=threads.index)
    provider_n = int(provider_threads.sum())

    diagnostics["hcw_provider_initiated_threads_n"] = provider_n
    diagnostics["hcw_provider_initiated_threads_pct"] = pct(provider_n, total_threads)

    if "INITIATOR_SENDER_TYPE" in threads.columns:
        initiator_type = threads["INITIATOR_SENDER_TYPE"].astype("string").str.lower()
        physician_app = provider_threads & initiator_type.str.contains(
            r"physician|advanced practice|app|nurse practitioner|physician assistant",
            case=False,
            na=False,
        )
        physician_app_n = int(physician_app.sum())
        nonphysician_app_n = int(provider_n - physician_app_n)
        diagnostics["physician_app_initiated_threads_n"] = physician_app_n
        diagnostics["physician_app_initiated_threads_pct"] = pct(physician_app_n, provider_n)
        diagnostics["nonphysician_app_provider_initiated_threads_n"] = nonphysician_app_n
        diagnostics["nonphysician_app_provider_initiated_threads_pct"] = pct(nonphysician_app_n, provider_n)
    else:
        missing_notes.append("INITIATOR_SENDER_TYPE/SENDER_TYPE")
        diagnostics["physician_app_initiated_threads_n"] = np.nan
        diagnostics["physician_app_initiated_threads_pct"] = np.nan
        diagnostics["nonphysician_app_provider_initiated_threads_n"] = np.nan
        diagnostics["nonphysician_app_provider_initiated_threads_pct"] = np.nan

    reply_required = {"thread_id", "CREATED_TIME", "origin", "thread_sequence_in_thread"}
    missing_reply_cols = sorted(reply_required - set(messages_threaded.columns))
    if missing_reply_cols:
        missing_notes.extend(missing_reply_cols)
        diagnostics["provider_initiated_threads_with_patient_reply_7d_n"] = np.nan
        diagnostics["provider_initiated_threads_with_patient_reply_7d_pct"] = np.nan
        diagnostics["median_time_to_first_patient_reply_hours"] = np.nan
    else:
        provider_thread_ids = set(threads.loc[provider_threads, "thread_id"].dropna().tolist())
        patient_replies = messages_threaded.loc[
            messages_threaded["thread_id"].isin(provider_thread_ids)
            & messages_threaded["origin"].eq("Patient")
            & messages_threaded["thread_sequence_in_thread"].gt(1),
            ["thread_id", "CREATED_TIME"],
        ]
        first_reply = (
            patient_replies.groupby("thread_id", dropna=False)["CREATED_TIME"]
            .min()
            .rename("FIRST_PATIENT_REPLY_TIME")
            .reset_index()
        )
        reply_threads = threads.loc[provider_threads, ["thread_id", "START_TIME"]].merge(first_reply, on="thread_id", how="left")
        reply_threads["time_to_first_patient_reply_hours"] = (
            reply_threads["FIRST_PATIENT_REPLY_TIME"].sub(reply_threads["START_TIME"]).dt.total_seconds().div(3600)
        )
        reply_within_7d = reply_threads["time_to_first_patient_reply_hours"].le(7 * 24)
        diagnostics["provider_initiated_threads_with_patient_reply_7d_n"] = int(reply_within_7d.sum())
        diagnostics["provider_initiated_threads_with_patient_reply_7d_pct"] = pct(int(reply_within_7d.sum()), provider_n)
        diagnostics["median_time_to_first_patient_reply_hours"] = float(reply_threads["time_to_first_patient_reply_hours"].median())

    diagnostics["missing_columns_or_notes"] = "; ".join(sorted(set(missing_notes))) if missing_notes else ""
    return diagnostics

analytic_relevant_diagnostics = pd.DataFrame(
    [
        analytic_relevant_diagnostics_for_cutoff(cutoff, result["messages"], result["threads"])
        for cutoff, result in thread_results.items()
    ]
).sort_values("cutoff_days")

analytic_relevant_diagnostics_path = RUN_DIR / "analytic_relevant_threading_diagnostics.csv"
analytic_relevant_diagnostics.to_csv(analytic_relevant_diagnostics_path, index=False)

missing_messages = analytic_relevant_diagnostics.loc[
    analytic_relevant_diagnostics["missing_columns_or_notes"].astype(str).str.len().gt(0),
    ["cutoff_days", "missing_columns_or_notes"],
]
if not missing_messages.empty:
    print("Some analytic-relevant diagnostics could not be fully calculated because columns were missing:")
    display(missing_messages)

display(analytic_relevant_diagnostics)

## 9. Save Primary 30-Day Threaded Handoff

The 30-day inactivity threshold is the primary threading definition used for the final analyses. This section saves a canonical working copy for downstream notebooks while preserving the timestamped run outputs above.

In [ ]:
messages_threaded_30d_path = RUN_DIR / "messages_threaded_30d.parquet"

if 30 in thread_results:
    messages_threaded_30d = thread_results[30]["messages"]
else:
    messages_threaded_30d = pd.read_parquet(messages_threaded_30d_path)

messages_threaded_30d.to_parquet(
    CANONICAL_THREADED_MESSAGES_PATH,
    index=False,
    compression="zstd",
)

handoff_summary = {
    "primary_cutoff_days": 30,
    "run_copy": str(messages_threaded_30d_path),
    "canonical_handoff_file": str(CANONICAL_THREADED_MESSAGES_PATH),
    "rows": int(len(messages_threaded_30d)),
    "threads": int(messages_threaded_30d["thread_id"].nunique()),
    "patients": int(messages_threaded_30d["DMRN"].nunique()),
}

print("Saved primary 30-day threaded handoff:")
print(json.dumps(handoff_summary, indent=2, default=str))

## 10. Output Index

List every file saved in this run folder and record the canonical downstream handoff path.

In [ ]:
def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return "not_installed"

output_index = pd.DataFrame(
    [
        {
            "file_name": path.name,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "modified_time": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
        }
        for path in sorted(RUN_DIR.iterdir())
        if path.is_file()
    ]
)
output_index_path = RUN_DIR / "output_index.csv"
output_index.to_csv(output_index_path, index=False)

reproducibility_summary = {
    "run_id": RUN_ID,
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "input_path": str(INPUT_PATH),
    "run_dir": str(RUN_DIR),
    "cutoff_days": CUTOFF_DAYS,
    "primary_cutoff_days": 30,
    "canonical_threaded_messages_path": str(CANONICAL_THREADED_MESSAGES_PATH),
    "python_version": sys.version,
    "platform": platform.platform(),
    "package_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": package_version("matplotlib"),
        "pyarrow": package_version("pyarrow"),
    },
    "privacy_restriction": "Output datasets are restricted EHR-derived records and should not be publicly shared. Share code and aggregate QA summaries only.",
}
reproducibility_summary_path = RUN_DIR / "threading_reproducibility_summary.json"
with open(reproducibility_summary_path, "w", encoding="utf-8") as f:
    json.dump(reproducibility_summary, f, indent=2, default=str)

# Refresh output index after writing reproducibility summary.
output_index = pd.DataFrame(
    [
        {
            "file_name": path.name,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "modified_time": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
        }
        for path in sorted(RUN_DIR.iterdir())
        if path.is_file()
    ]
)
output_index.to_csv(output_index_path, index=False)

print("Saved files in run folder:")
display(output_index)
print("RUN_DIR:", RUN_DIR)
print("Canonical threaded message handoff:", CANONICAL_THREADED_MESSAGES_PATH)